## Data

- To clean the data, I first removed the percentage symbols % from `host_response_rate` and `host_acceptance_rate` and converted them to floats. They were initially stored as strings, so converting them to a numeric value would make it usable by the XGBoost model. Beyond the formatting issue, I thought that these features would be strong predictors of superhost status. After looking into Airbnb, I learned that they actually require superhosts to maintain a high response rate of 90% or higher, so that further solidified the idea that these predictors are strong signals in superhost eligibility.
- I also extracted the numeric value from `bathrooms_text`, as it was stored as "1.5 baths" or "1 private bath". I choose to do this so it would also be easier for the model to use a numeric value instead of one-hot-encoding into multiple unique columns or dropping the variable altogether, which I wanted to avoid since I believe the number of bathrooms is very informative in relation to the response.
- For **missing values**, I imputed numerical variables using the training data's mean and applied this to both the training and test datasets. I chose to use the training mean to try and preserve the overall distribution of the feature without introducing extreme values.
- For categorical variables, I filled missing values with a "missing" category so that the model treated these observations as its own category rather than discarding those observations. I believed that the fact that these observations have missing values doesn't mean that they were uninformative (e.g. a host with no host_response_time recorded may genuinely be an infrequent responder, which is meaningful information). Treating missing values as its own category lets the model include observations with missing values and learn from its patterns without fully obscuring it.
- I dropped free-text variables such as `description` and `host_about` because I did not think they would be helpful in predicting the response for the purposes of this project. I also decided to drop `host_neighbourhood` and `host_location` because I thought that these variables provided redundant information with other location-related features such as `listing_location` and `neighbourhood_cleansed`.
- Finally, since XGBoost does not handle string inputs naturally, I label encoded the categorical variables, converting each unique category to an integer. One complication with label encoding is that the test set may contain category values that never appeared in the training set, which would cause the encoder to throw an error. To handle this, I mapped any unseen test categories to a known training category before encoding, ensuring the pipeline runs without errors on new data.


## Model

- I began by fitting an **XGBoost** classifier as my model as my first choice. XGBoost is a gradient boosting method, which means that it builds an ensemble of trees sequentially, where each new tree is trained to correct the errors of the trees before it. This model contrasts to bagging-based methods like random forests and bagged trees, which builds trees independently in parallel and **average** their predictions. Additionally, models like KNN would have required additional feature scaling and general preprocessing steps.
- My initial submission utilized a set of reasonable starting hyperparameters to input in the XGBoost model. 500 estimators, learning rate of 0.05, max depth of 6, and a subsample rate of 0.8, which returned a score of 0.9504 on Kaggle. Even though this surpassed the required threshold, I decided to tune these hyperparameters using `RandomizedSearchCV` using 3-fold stratified cross-validation. Although this method does not evaluate every possible combination like GridSearchCV, it samples the search space efficiently and is a reasonable approach when the grid is large, so I decided on using RandomizedSearchCV to preserve computational resources. I also passed the threshold at this point so I wanted to add more reliable hyperparameters in my model rather than relying solely on my own judgment for the final hyperparameter values.
    - For the first search, I started with a wide range of n_estimators, max_depth, learning_rate, and subsample, evaluating on the `auc` metric. Many of the values in the output were in the upper or lower edges of my range, so I conducted another search shifting my range and zooming in on hyperparameters that have landed somewhere in the middle of the range to get a more precise number.
    - I continued conducting additional searches until my outputs landed somewhere in the middle of the ranges that I defined in my grid, which took 4 searches.
    - I then plugged in these hyperparameter values in my submission template and submitted the predictions onto Kaggle, where my score turned out to be 0.9560.

Additionally, for all of my searches and the creation of my XGBoost model, I set `random_state = 16` for reproducibility and consistency.

## Runtime

- My submitted code took **20 seconds** to run all cells.
- I ran the code through **Jupyter Notebook in Anaconda**.